In [1]:
using LatticeDiracOperators
using Gaugefields
using LMP
using LinearAlgebra

NX = 4
NY = 4
NZ = 4
NT = 4
Nwing = 0
NC = 3

U_cpu = Initialize_Gaugefields(NC, Nwing, NX, NY, NZ, NT, condition="hot")
x_test = Initialize_pseudofermion_fields(U_cpu[1], "Wilson", nowing=true)
gauss_distribution_fermion!(x_test)

println("Setup done: $(NX)×$(NY)×$(NZ)×$(NT) lattice, NC=$NC")

┌ Warning: Circular dependency detected. Precompilation will be skipped for:
│   ChainRules [082447d4-558c-5d27-93f4-14fc19e9eca2]
│   SpecialFunctionsExt [1285c0f1-ff9f-5867-b66e-0f359bcf09ba]
│   GPUArrays [0c68f7d7-f131-5f86-a1c3-88cf8149b2d7]
│   LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018]
│   KernelAbstractions [63c18a36-062a-441e-b654-da1e3ab1ce7c]
│   CUDA [052768ef-5323-5732-b1bb-66c8b64840ba]
│   SparseArraysExt [85068d23-b5fb-53f1-8204-05c2aba6942f]
│   StructArraysGPUArraysCoreExt [cbbf52aa-d660-55d4-8a5a-33845bbaa85d]
│   AtomixCUDAExt [13011619-4c7c-5ef0-948f-5fc81565cd05]
│   LinearAlgebraExt [66d79d19-2cc4-5b0b-ac7a-b340256d1ecd]
│   ZygoteColorsExt [e68c091a-8ea5-5ca7-be4f-380657d4ad79]
│   ChainRulesCoreExt [eae2faf6-b232-58cb-a410-7764fda2830c]
│   Zygote [e88e6eb3-aa80-5325-afca-941959d7151f]
└ @ Pkg.API C:\Users\_s2220186\AppData\Local\Programs\Julia-1.10.7\share\julia\stdlib\v1.10\Pkg\src\API.jl:1279
[ Info: Precompiling LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018]
[ 

Setup done: 4×4×4×4 lattice, NC=3


In [2]:
# ================================================================
# NormLayer DeltaRule! 수치 검증
#
# Forward:  y_i = alpha * x_i / ||x_i||
# Loss:     L = Re( inner_product(d, conj(y)) )
# DeltaRule convention in LMP: δ = dL/dRe(x) - i·dL/dIm(x)
#   (= ∂L/∂x  in Wirtinger sense, NOT ∂L/∂x*)
# Numeric FD must use the same convention.
# ================================================================

norm_layer = NormLayer()

# 업스트림 delta d — 랜덤 복소수 fermion
d_test = deepcopy(x_test)
gauss_distribution_fermion!(d_test)

# --- Analytic DeltaRule! ---
WF3x1 = Union{LMP.Layer_types.WF3x1CPU, LMP.Layer_types.WF3x1GPU}
z_vec     = Vector{WF3x1}([x_test])
d_vec     = Vector{WF3x1}([d_test])
delta_vec = Vector{WF3x1}([deepcopy(x_test)])

LMP.DeltaRule!(norm_layer, Identity(), z_vec, d_vec, delta_vec)
delta_analytic = delta_vec[1]

# --- 수치 미분 (finite difference) ---
function loss_norm(x_in)
    y = norm_layer([x_in])
    return real(inner_product(d_test, conj(y[1])))
end

eps_fd = 1e-5

NX_ = x_test.NX; NY_ = x_test.NY; NZ_ = x_test.NZ
NT_ = x_test.NT; NC_ = x_test.NC
delta_numeric = deepcopy(x_test)

for it in 1:NT_, iz in 1:NZ_, iy in 1:NY_, ix in 1:NX_, ic in 1:NC_
    for ig in 1:4
        v0 = x_test[ic, ix, iy, iz, it, ig]

        x_test[ic, ix, iy, iz, it, ig] = v0 + eps_fd
        Lp = loss_norm(x_test)
        x_test[ic, ix, iy, iz, it, ig] = v0 - eps_fd
        Lm = loss_norm(x_test)
        dL_dRe = (Lp - Lm) / (2eps_fd)

        x_test[ic, ix, iy, iz, it, ig] = v0 + im*eps_fd
        Lp_im = loss_norm(x_test)
        x_test[ic, ix, iy, iz, it, ig] = v0 - im*eps_fd
        Lm_im = loss_norm(x_test)
        dL_dIm = (Lp_im - Lm_im) / (2eps_fd)

        # LMP convention: δ = dL/dRe - i·dL/dIm  (Wirtinger ∂/∂x, not ∂/∂x*)
        delta_numeric[ic, ix, iy, iz, it, ig] = dL_dRe - im*dL_dIm

        x_test[ic, ix, iy, iz, it, ig] = v0  # 복구
    end
end

# --- 비교 ---
diff     = delta_analytic - delta_numeric
abs_err  = sqrt(real(inner_product(diff, diff)))
ref_norm = sqrt(real(inner_product(delta_numeric, delta_numeric)))
rel_err  = abs_err / ref_norm

println("=" ^ 50)
println("  NormLayer DeltaRule! 수치 검증")
println("=" ^ 50)
println("  |analytic - numeric|  = ", round(abs_err,  sigdigits=4))
println("  |numeric|             = ", round(ref_norm, sigdigits=4))
println("  relative error        = ", round(rel_err,  sigdigits=4))
if rel_err < 1e-4
    println("  ✓  PASS  (rel error < 1e-4)")
else
    println("  ✗  FAIL  (rel error too large)")
end

  NormLayer DeltaRule! 수치 검증
  |analytic - numeric|  = 7.417e-9
  |numeric|             = 1.019
  relative error        = 7.279e-9
  ✓  PASS  (rel error < 1e-4)


In [3]:
# --- 성분별 상세 비교 (처음 20개 성분) ---
println("\n  idx   analytic (Re/Im)          numeric (Re/Im)            abs_diff")
println("  ", "-" ^ 80)

count = 0
outer = false
for it in 1:NT_, iz in 1:NZ_, iy in 1:NY_, ix in 1:NX_, ic in 1:NC_
    outer && break
    for ig in 1:4
        count += 1
        if count > 20
            outer = true
            break
        end
        a_val = delta_analytic[ic, ix, iy, iz, it, ig]
        n_val = delta_numeric[ic, ix, iy, iz, it, ig]
        d_val = abs(a_val - n_val)
        println("  $(count)   ($(round(real(a_val),sigdigits=6)), $(round(imag(a_val),sigdigits=6)))   ($(round(real(n_val),sigdigits=6)), $(round(imag(n_val),sigdigits=6)))   $(round(d_val,sigdigits=4))")
    end
end
println("  (이후 성분 생략)")


  idx   analytic (Re/Im)          numeric (Re/Im)            abs_diff
  --------------------------------------------------------------------------------
  1   (0.0139034, -0.00738746)   (0.0139034, -0.00738746)   1.828e-10
  2   (0.0327584, 0.00910882)   (0.0327584, 0.00910882)   1.488e-10
  3   (0.0127088, 0.00211817)   (0.0127088, 0.00211817)   1.169e-10
  4   (-0.0222761, 0.00392939)   (-0.0222761, 0.00392939)   1.54e-10
  5   (0.0122401, 0.000589265)   (0.0122401, 0.000589265)   1.253e-10
  6   (0.00598976, 0.0137998)   (0.00598976, 0.0137998)   1.501e-10
  7   (0.000864899, 0.0158106)   (0.000864899, 0.0158106)   1.139e-10
  8   (-0.00691947, -0.000424257)   (-0.00691947, -0.000424257)   1.889e-10
  9   (0.000843929, -0.0237882)   (0.000843929, -0.0237882)   2.164e-10
  10   (-0.00238353, -0.0116463)   (-0.00238353, -0.0116463)   1.528e-10
  11   (-0.00691821, 0.0112135)   (-0.00691821, 0.0112135)   8.777e-11
  12   (-0.0120375, 0.000132825)   (-0.0120375, 0.000132825)   8.866e-1